
# Misogyny Detection — Cascaded Classification Pipeline

**Task A** (binary): Is the meme misogynous?  
**Task B** (multi-label): Which sub-types apply? *(shaming, stereotype, objectification, violence)*

Pipeline: **Stage 1** classifies all samples → **Stage 2** runs *only* on predicted-misogynous samples → end-to-end evaluation combines both stages.


In [1]:

import os, time, json, re, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Transformers
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

# ── Dataset column constants ─────────────────────────────────────────────────
fixed_cols = ['file_name', 'misogynous', 'shaming', 'stereotype', 'objectification', 'violence']
label_cols  = ['shaming', 'stereotype', 'objectification', 'violence']


Device: cuda
PyTorch version: 2.10.0+cu130


In [2]:

# ── Model selection ───────────────────────────────────────────────────────────
# cardiffnlp/twitter-roberta-base-hate: RoBERTa fine-tuned on ~124M tweets +
# HatEval/OffComEval hate-speech data.  Domain match with meme captions is key:
# same informal register, abbreviations, and irony patterns.
BERT_MODEL   = 'cardiffnlp/twitter-roberta-base-hate'
BERT_MAX_LEN = 128
BERT_BATCH   = 32

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)


class BertDataset(Dataset):
    """Tokenized dataset — used for inference / standalone evaluation."""
    def __init__(self, texts, labels=None, max_len=BERT_MAX_LEN):
        self.enc = tokenizer(texts, truncation=True, padding=True,
                             max_length=max_len, return_tensors='pt')
        self.labels = labels

    def __len__(self):
        return self.enc['input_ids'].size(0)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        if self.labels is not None:
            lbl = self.labels[idx]
            dtype = torch.float if isinstance(lbl, list) else torch.long
            item['labels'] = torch.tensor(lbl, dtype=dtype)
        return item


class MultiTaskDataset(Dataset):
    """Returns binary + multi-label targets for each sample (used in joint training)."""
    def __init__(self, texts, binary_labels, ml_labels, max_len=BERT_MAX_LEN):
        self.enc = tokenizer(texts, truncation=True, padding=True,
                             max_length=max_len, return_tensors='pt')
        self.bin = binary_labels
        self.ml  = ml_labels

    def __len__(self):
        return self.enc['input_ids'].size(0)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['binary_label'] = torch.tensor(self.bin[idx], dtype=torch.long)
        item['ml_labels']    = torch.tensor(self.ml[idx],  dtype=torch.float)
        return item


class MultiTaskModel(nn.Module):
    """
    Shared RoBERTa backbone  +  binary classification head  +  multi-label head.
    Single forward pass → (binary_logits, ml_logits).
    Training: binary CE on ALL samples, ML BCE only on misogynous samples.
    This lets the backbone learn from 7,500 samples instead of 2,655.
    """
    def __init__(self, model_name, n_ml_labels=4):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        h  = config.hidden_size
        dp = getattr(config, 'classifier_dropout', None) or config.hidden_dropout_prob
        # Replicate RoBERTa classification-head architecture: dense → tanh → proj
        self.dropout   = nn.Dropout(dp)
        self.bin_dense = nn.Linear(h, h)
        self.bin_proj  = nn.Linear(h, 2)
        self.ml_dense  = nn.Linear(h, h)
        self.ml_proj   = nn.Linear(h, n_ml_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.backbone(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state[:, 0]
        # Binary head
        b = self.dropout(cls)
        b = torch.tanh(self.bin_dense(b))
        b = self.dropout(b)
        # Multi-label head
        m = self.dropout(cls)
        m = torch.tanh(self.ml_dense(m))
        m = self.dropout(m)
        return self.bin_proj(b), self.ml_proj(m)


In [3]:

# AMP: fp16 forward pass on Tensor Cores (~2-3x faster on RTX)
_USE_AMP = DEVICE.type == 'cuda'
print(f'AMP (fp16): {"ON" if _USE_AMP else "OFF"}')


# ══════════════════════════════════════════════════════════════════════════════
# FGM — Fast Gradient Method adversarial training
# Perturbs word-embedding weights in the gradient direction to produce
# adversarial examples on-the-fly.  Free +1-3% F1 at negligible cost.
# ══════════════════════════════════════════════════════════════════════════════
class FGM:
    def __init__(self, model, epsilon=1.0, emb_name='word_embeddings'):
        self.model, self.epsilon, self.emb_name = model, epsilon, emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if self.emb_name in name and param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isinf(norm):
                    param.data.add_(self.epsilon * param.grad / norm)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


def _make_param_groups(model, lr):
    """
    Layer-wise LR decay — model-agnostic.
    Head parameters get `lr`, backbone gets `lr / 10`.
    """
    backbone_prefixes = ('backbone.', 'roberta.', 'distilbert.', 'bert.', 'deberta.')
    backbone_pfx = next(
        (pfx for pfx in backbone_prefixes
         if any(n.startswith(pfx) for n, _ in model.named_parameters())),
        None,
    )
    no_decay = {'bias', 'LayerNorm.weight', 'layer_norm.weight'}
    groups = []
    for is_backbone, head_lr in [(False, lr), (True, lr / 10)]:
        for no_dec, wd in [(False, 0.01), (True, 0.0)]:
            params = [
                p for n, p in model.named_parameters()
                if (backbone_pfx is not None and n.startswith(backbone_pfx)) == is_backbone
                and any(nd in n for nd in no_decay) == no_dec
            ]
            if params:
                groups.append({'params': params, 'lr': head_lr, 'weight_decay': wd})
    return groups


# ══════════════════════════════════════════════════════════════════════════════
# Multi-task joint training:
#   - Binary CE loss on ALL samples
#   - Multi-label BCE loss only on misogynous samples
#   - FGM adversarial perturbation every step
#   - Validation: cascaded binary→multi-label combined F1
# ══════════════════════════════════════════════════════════════════════════════
def train_multitask(
    model, train_loader, val_loader, *,
    binary_loss_fn, ml_loss_fn, ml_weight=1.0,
    lr=3e-5, epochs=25, patience=8, warmup_ratio=0.1,
    grad_accum=2, fgm_epsilon=1.0,
):
    n_updates    = (len(train_loader) + grad_accum - 1) // grad_accum
    total_steps  = n_updates * epochs
    warmup_steps = int(total_steps * warmup_ratio)

    optimizer = torch.optim.AdamW(_make_param_groups(model, lr))
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler    = torch.cuda.amp.GradScaler(enabled=_USE_AMP)
    fgm       = FGM(model, epsilon=fgm_epsilon)

    best_val   = 0.0
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    no_improve = 0
    history    = {'val_bin_f1': [], 'val_ml_f1': [], 'val_combined': []}
    t0         = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader, 1):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            blab = batch['binary_label'].to(DEVICE)
            mlab = batch['ml_labels'].to(DEVICE)

            # ── Forward + loss ────────────────────────────────────
            with torch.cuda.amp.autocast(enabled=_USE_AMP):
                bin_logits, ml_logits = model(ids, mask)
                loss_bin = binary_loss_fn(bin_logits, blab)
                mis = blab == 1
                loss_ml = ml_loss_fn(ml_logits[mis], mlab[mis]) if mis.any() else ids.new_tensor(0.0, dtype=torch.float)
                loss = (loss_bin + ml_weight * loss_ml) / grad_accum
            scaler.scale(loss).backward()

            # ── FGM adversarial step ──────────────────────────────
            fgm.attack()
            with torch.cuda.amp.autocast(enabled=_USE_AMP):
                ba, ma = model(ids, mask)
                adv = binary_loss_fn(ba, blab)
                if mis.any():
                    adv = adv + ml_weight * ml_loss_fn(ma[mis], mlab[mis])
                adv = adv / grad_accum
            scaler.scale(adv).backward()
            fgm.restore()

            if step % grad_accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        # ── Validation (cascaded binary → multi-label) ────────────
        model.eval()
        bp_v, bl_v, mp_v, ml_v = [], [], [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                with torch.cuda.amp.autocast(enabled=_USE_AMP):
                    bv, mv = model(ids, mask)
                bp_v.extend(bv.argmax(1).cpu().tolist())
                bl_v.extend(batch['binary_label'].tolist())
                mp_v.append(torch.sigmoid(mv).cpu())
                ml_v.extend(batch['ml_labels'].tolist())

        val_bin = f1_score(bl_v, bp_v, average='macro')
        probs   = torch.cat(mp_v, dim=0).numpy()
        ml_pred = [
            [int(probs[i, k] > 0.5) for k in range(probs.shape[1])] if bp == 1 else [0]*probs.shape[1]
            for i, bp in enumerate(bp_v)
        ]
        val_ml  = f1_score(ml_v, ml_pred, average='macro')
        val_c   = 0.4 * val_bin + 0.6 * val_ml

        history['val_bin_f1'].append(val_bin)
        history['val_ml_f1'].append(val_ml)
        history['val_combined'].append(val_c)
        print(f'  Epoch {epoch:02d} | bin={val_bin:.4f}  ml={val_ml:.4f}  comb={val_c:.4f}  best={best_val:.4f}')

        if val_c > best_val:
            best_val   = val_c
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return history, time.time() - t0


@torch.no_grad()
def tune_ml_thresholds(model, texts, labels, n_steps=80):
    """Per-label threshold search on misogynous validation subset using MTL model's ML head."""
    model.eval()
    ds     = BertDataset(texts)
    loader = DataLoader(ds, batch_size=BERT_BATCH, shuffle=False, num_workers=0)
    all_probs = []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=_USE_AMP):
            _, ml_logits = model(**batch)
        all_probs.append(torch.sigmoid(ml_logits).cpu())
    probs = torch.cat(all_probs, dim=0).numpy()
    y     = np.array(labels)
    best_thrs = []
    for i in range(probs.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in np.linspace(0.1, 0.9, n_steps):
            f1 = f1_score(y[:, i], (probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        best_thrs.append(best_t)
        print(f'  label[{i}]: best_thr={best_t:.3f}  val_F1={best_f1:.4f}')
    return best_thrs


AMP (fp16): ON



## 1. Data Loading & Unified Split
One stratified split shared by both stages — no data leakage, directly comparable metrics.


In [4]:

# ══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING — UNIFIED SPLIT
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_csv('data/training/training.csv', sep='\t', header=0)
text_cols = df.columns[len(fixed_cols):]
df['Text'] = df[text_cols].astype(str).agg(' '.join, axis=1)
df = df[fixed_cols + ['Text']]

N_EXAMPLES = 7500
df = df.iloc[:N_EXAMPLES].reset_index(drop=True)

X_all     = df['Text']
y_bin_all = df['misogynous']
y_ml_all  = df[label_cols]

# 80/20 → train+val / test  (stratified by binary label)
X_tv, X_test, y_bin_tv, y_bin_test, y_ml_tv, y_ml_test = train_test_split(
    X_all, y_bin_all, y_ml_all,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_bin_all
)

# 87.5/12.5 → train / val
X_train, X_val, y_bin_train, y_bin_val, y_ml_train, y_ml_val = train_test_split(
    X_tv, y_bin_tv, y_ml_tv,
    test_size=0.125, random_state=RANDOM_STATE, stratify=y_bin_tv
)

# ── List conversions ──────────────────────────────────────────────────────────
X_train_list     = X_train.tolist()
X_val_list       = X_val.tolist()
X_test_list      = X_test.tolist()

y_bin_train_list = y_bin_train.tolist()
y_bin_val_list   = y_bin_val.tolist()
y_bin_test_list  = y_bin_test.tolist()

# ALL samples (non-misogynous have [0,0,0,0]) — used for multi-task training
y_ml_train_list  = y_ml_train.values.tolist()
y_ml_val_list    = y_ml_val.values.tolist()
y_ml_test_list   = y_ml_test.values.tolist()

# ── Misogynous-only subsets (for threshold tuning on ML head) ─────────────────
mis_train = y_bin_train == 1
mis_val   = y_bin_val == 1

X_train_ml_list = X_train[mis_train].tolist()
y_train_ml_list = y_ml_train[mis_train].values.tolist()
X_val_ml_list   = X_val[mis_val].tolist()
y_val_ml_list   = y_ml_val[mis_val].values.tolist()

print('Unified split:')
print(f'  Train: {len(X_train_list):,}  |  Val: {len(X_val_list):,}  |  Test: {len(X_test_list):,}')
print(f'  Binary train balance: {dict(y_bin_train.value_counts().sort_index())}')
print(f'  ML train (all): {len(y_ml_train_list):,}  |  ML train (mis only): {len(X_train_ml_list):,}')
print(f'  ML val   (mis only): {len(X_val_ml_list):,}')


Unified split:
  Train: 5,250  |  Val: 750  |  Test: 1,500
  Binary train balance: {0: np.int64(2595), 1: np.int64(2655)}
  ML train (all): 5,250  |  ML train (mis only): 2,655
  ML val   (mis only): 379



## 2. Non-DL Baseline — TF-IDF + Logistic Regression (Cascaded)


In [5]:

# ══════════════════════════════════════════════════════════════════════════════
# 2. NON-DL BASELINE — TF-IDF + LOGISTIC REGRESSION (CASCADED)
# ══════════════════════════════════════════════════════════════════════════════

# ── Stage 1: Binary baseline ─────────────────────────────────────────────────
binary_baseline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced',
                                  random_state=RANDOM_STATE)),
])
binary_baseline.fit(X_train_list, y_bin_train_list)
baseline_bin_preds = binary_baseline.predict(X_test_list)
baseline_bin_f1    = f1_score(y_bin_test_list, baseline_bin_preds, average='macro')

print('── TF-IDF + LR Baseline — Stage 1 (Binary) ─────────────────────────────')
print(f'  F1-Macro : {baseline_bin_f1:.4f}')
print(classification_report(y_bin_test_list, baseline_bin_preds,
                            target_names=['non-misogynous', 'misogynous']))

# ── Stage 2: Multi-label baseline (trained on misogynous train subset only) ──
ml_baseline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   MultiOutputClassifier(
        LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced',
                           random_state=RANDOM_STATE), n_jobs=-1)),
])
ml_baseline.fit(X_train_ml_list, y_train_ml_list)

# ── End-to-end cascaded baseline: binary → filter → multi-label ──────────────
baseline_cascaded_preds = []
for text, bp in zip(X_test_list, baseline_bin_preds):
    if bp == 1:
        baseline_cascaded_preds.append(ml_baseline.predict([text])[0].tolist())
    else:
        baseline_cascaded_preds.append([0, 0, 0, 0])

baseline_ml_f1 = f1_score(y_ml_test_list, baseline_cascaded_preds, average='macro')

print('── TF-IDF + LR Baseline — End-to-End Cascaded Pipeline ─────────────────')
print(f'  E2E F1-Macro : {baseline_ml_f1:.4f}')
print(classification_report(y_ml_test_list, baseline_cascaded_preds,
                            target_names=label_cols))


── TF-IDF + LR Baseline — Stage 1 (Binary) ─────────────────────────────
  F1-Macro : 0.7879
                precision    recall  f1-score   support

non-misogynous       0.77      0.82      0.79       742
    misogynous       0.81      0.76      0.78       758

      accuracy                           0.79      1500
     macro avg       0.79      0.79      0.79      1500
  weighted avg       0.79      0.79      0.79      1500

── TF-IDF + LR Baseline — End-to-End Cascaded Pipeline ─────────────────
  E2E F1-Macro : 0.4829
                 precision    recall  f1-score   support

        shaming       0.46      0.37      0.41       206
     stereotype       0.62      0.62      0.62       458
objectification       0.54      0.50      0.52       346
       violence       0.38      0.39      0.39       163

      micro avg       0.53      0.51      0.52      1173
      macro avg       0.50      0.47      0.48      1173
   weighted avg       0.53      0.51      0.52      1173
    samples a


## 3. Multi-Task Model Training (Binary + Multi-label with FGM)
Shared RoBERTa backbone trained jointly on **both tasks** — the backbone sees all 7,500 samples. FGM adversarial training perturbs embeddings each step for extra robustness.


In [6]:

# ══════════════════════════════════════════════════════════════════════════════
# 3. MULTI-TASK MODEL — DATA LOADERS & LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

mtl_train_ds = MultiTaskDataset(X_train_list, y_bin_train_list, y_ml_train_list)
mtl_val_ds   = MultiTaskDataset(X_val_list,   y_bin_val_list,   y_ml_val_list)

mtl_train_loader = DataLoader(mtl_train_ds, batch_size=BERT_BATCH, shuffle=True,  num_workers=0, pin_memory=True)
mtl_val_loader   = DataLoader(mtl_val_ds,   batch_size=BERT_BATCH, shuffle=False, num_workers=0, pin_memory=True)

# ── Binary loss (weighted CE) ─────────────────────────────────────────────────
pos_count = sum(y_bin_train_list)
neg_count = len(y_bin_train_list) - pos_count
binary_class_weights = torch.tensor([1.0, neg_count / pos_count], device=DEVICE)
binary_loss_fn = nn.CrossEntropyLoss(weight=binary_class_weights)

# ── Multi-label loss (weighted BCE per label, only computed on misogynous) ────
y_train_ml_tensor = torch.tensor(y_train_ml_list, dtype=torch.float)
pos_counts  = y_train_ml_tensor.sum(dim=0).clamp(min=1)
neg_counts  = len(y_train_ml_tensor) - pos_counts
pos_weights = (neg_counts / pos_counts).to(DEVICE)
ml_loss_fn  = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print(f'Multi-task DataLoaders ready')
print(f'  Train: {len(X_train_list):,}  |  Val: {len(X_val_list):,}')
print(f'  Binary class weights: [1.000, {binary_class_weights[1]:.3f}]')
print(f'  ML pos_weight (mis-only): {pos_weights.cpu().round(decimals=2).tolist()}')


Multi-task DataLoaders ready
  Train: 5,250  |  Val: 750
  Binary class weights: [1.000, 0.977]
  ML pos_weight (mis-only): [2.8499999046325684, 0.6499999761581421, 1.190000057220459, 3.7799999713897705]


In [7]:

print(f'Initialising multi-task model ({BERT_MODEL})…')
mtl_model = MultiTaskModel(BERT_MODEL, n_ml_labels=len(label_cols)).to(DEVICE)

n_params = sum(p.numel() for p in mtl_model.parameters() if p.requires_grad)
print(f'  Trainable parameters: {n_params:,}')

print('\nJoint training (binary + multi-label + FGM adversarial)…')
history, train_time = train_multitask(
    mtl_model, mtl_train_loader, mtl_val_loader,
    binary_loss_fn=binary_loss_fn, ml_loss_fn=ml_loss_fn,
    ml_weight=1.0, lr=3e-5, epochs=30, patience=8,
    warmup_ratio=0.1, grad_accum=2, fgm_epsilon=1.0,
)

print(f'\n── Multi-task Training Complete ────────────────────────────────────────')
print(f'  Total time: {train_time:.1f}s')
print(f'  Best combined val: {max(history["val_combined"]):.4f}')
print(f'  Best bin val F1:   {max(history["val_bin_f1"]):.4f}')
print(f'  Best ml  val F1:   {max(history["val_ml_f1"]):.4f}')


Initialising multi-task model (cardiffnlp/twitter-roberta-base-hate)…


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate
Key                             | Status     | 
--------------------------------+------------+-
classifier.dense.weight         | UNEXPECTED | 
classifier.dense.bias           | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | UNEXPECTED | 
classifier.out_proj.bias        | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Trainable parameters: 125,831,430

Joint training (binary + multi-label + FGM adversarial)…
  Epoch 01 | bin=0.6643  ml=0.2638  comb=0.4240  best=0.0000
  Epoch 02 | bin=0.7074  ml=0.3523  comb=0.4943  best=0.4240
  Epoch 03 | bin=0.7573  ml=0.3848  comb=0.5338  best=0.4943
  Epoch 04 | bin=0.7733  ml=0.4438  comb=0.5756  best=0.5338
  Epoch 05 | bin=0.7773  ml=0.4515  comb=0.5818  best=0.5756
  Epoch 06 | bin=0.7903  ml=0.4805  comb=0.6044  best=0.5818
  Epoch 07 | bin=0.7827  ml=0.4816  comb=0.6020  best=0.6044
  Epoch 08 | bin=0.8039  ml=0.4999  comb=0.6215  best=0.6044
  Epoch 09 | bin=0.7999  ml=0.5011  comb=0.6206  best=0.6215
  Epoch 10 | bin=0.7972  ml=0.4970  comb=0.6171  best=0.6215
  Epoch 11 | bin=0.8039  ml=0.5061  comb=0.6253  best=0.6215
  Epoch 12 | bin=0.8038  ml=0.5138  comb=0.6298  best=0.6253
  Epoch 13 | bin=0.8093  ml=0.5048  comb=0.6266  best=0.6298
  Epoch 14 | bin=0.8160  ml=0.4967  comb=0.6244  best=0.6298
  Epoch 15 | bin=0.8147  ml=0.5194  comb=0.6375  bes

In [8]:

# ── Threshold tuning on misogynous validation subset ──────────────────────────
# The multi-label head outputs raw logits.  We search for per-label thresholds
# that maximise F1 on the misogynous-only validation split.

print('Tuning per-label thresholds on misogynous validation set…')
ml_thresholds = tune_ml_thresholds(mtl_model, X_val_ml_list, y_val_ml_list)
print(f'\nOptimal thresholds: {[round(t, 3) for t in ml_thresholds]}')


Tuning per-label thresholds on misogynous validation set…
  label[0]: best_thr=0.465  val_F1=0.5918
  label[1]: best_thr=0.100  val_F1=0.7675
  label[2]: best_thr=0.292  val_F1=0.6812
  label[3]: best_thr=0.586  val_F1=0.5818

Optimal thresholds: [0.465, 0.1, 0.292, 0.586]



## 4. Cascaded Pipeline Evaluation
Binary head predicts on **all** test samples → multi-label head runs **only** on predicted-misogynous → end-to-end F1.


In [9]:

# ══════════════════════════════════════════════════════════════════════════════
# 4. CASCADED PIPELINE EVALUATION — BINARY → MULTI-LABEL
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def cascaded_predict(model, texts, thresholds):
    """
    Full cascaded pipeline using the multi-task model.
    One forward pass → binary head decides misogynous/not →
    multi-label head predictions kept only for predicted-misogynous.
    """
    model.eval()
    ds     = BertDataset(texts)
    loader = DataLoader(ds, batch_size=BERT_BATCH, shuffle=False, num_workers=0)

    all_bin, all_probs = [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=_USE_AMP):
            bin_logits, ml_logits = model(**batch)
        all_bin.extend(bin_logits.argmax(1).cpu().tolist())
        all_probs.append(torch.sigmoid(ml_logits).cpu())

    probs = torch.cat(all_probs, dim=0).numpy()
    ml_preds = []
    for i, bp in enumerate(all_bin):
        if bp == 1:
            ml_preds.append([int(probs[i, k] > thresholds[k]) for k in range(len(thresholds))])
        else:
            ml_preds.append([0, 0, 0, 0])

    return all_bin, ml_preds


# ── Run cascaded evaluation on test set ───────────────────────────────────────
t_inf = time.time()
cascade_bin_preds, cascade_ml_preds = cascaded_predict(
    mtl_model, X_test_list, ml_thresholds
)
cascaded_inf_time = (time.time() - t_inf) / len(X_test_list) * 1000

cascaded_bin_f1 = f1_score(y_bin_test_list, cascade_bin_preds, average='macro')
cascaded_ml_f1  = f1_score(y_ml_test_list, cascade_ml_preds, average='macro')

print('══════════════════════════════════════════════════════════════════════════')
print('  CASCADED PIPELINE RESULTS  (Multi-Task + FGM)')
print('══════════════════════════════════════════════════════════════════════════')
print(f'  Binary F1-Macro         : {cascaded_bin_f1:.4f}')
print(f'  End-to-End ML F1-Macro  : {cascaded_ml_f1:.4f}')
print(f'  Inference speed         : {cascaded_inf_time:.3f} ms/sample')
print()
print('── Stage 1: Binary ─────────────────────────────────────────────────────')
print(classification_report(
    y_bin_test_list, cascade_bin_preds,
    target_names=['non-misogynous', 'misogynous']
))
print('── End-to-End: Multi-label (cascaded) ──────────────────────────────────')
print(classification_report(
    y_ml_test_list, cascade_ml_preds,
    target_names=label_cols
))


══════════════════════════════════════════════════════════════════════════
  CASCADED PIPELINE RESULTS  (Multi-Task + FGM)
══════════════════════════════════════════════════════════════════════════
  Binary F1-Macro         : 0.8253
  End-to-End ML F1-Macro  : 0.5199
  Inference speed         : 1.058 ms/sample

── Stage 1: Binary ─────────────────────────────────────────────────────
                precision    recall  f1-score   support

non-misogynous       0.81      0.85      0.83       742
    misogynous       0.84      0.81      0.82       758

      accuracy                           0.83      1500
     macro avg       0.83      0.83      0.83      1500
  weighted avg       0.83      0.83      0.83      1500

── End-to-End: Multi-label (cascaded) ──────────────────────────────────
                 precision    recall  f1-score   support

        shaming       0.36      0.58      0.45       206
     stereotype       0.52      0.83      0.64       458
objectification       0.48    


## 5. Save Models & Export (PyTorch + ONNX)


In [10]:

SAVE_DIR = os.path.join('models', 'trained')
os.makedirs(SAVE_DIR, exist_ok=True)

# ── PyTorch weights (single multi-task model) ────────────────────────────────
torch.save(mtl_model.state_dict(), os.path.join(SAVE_DIR, 'multitask_model_weights.pt'))
print(f'Weights saved to {SAVE_DIR}/')

# ── Thresholds ────────────────────────────────────────────────────────────────
with open(os.path.join(SAVE_DIR, 'ml_thresholds.json'), 'w') as f:
    json.dump(ml_thresholds, f)
print(f'Thresholds saved: {ml_thresholds}')

# ── ONNX export (both heads in one graph) ────────────────────────────────────
class _OnnxWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask):
        bin_logits, ml_logits = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return bin_logits, ml_logits

def export_onnx(model, name):
    cpu_model = model.cpu().eval()
    wrapper   = _OnnxWrapper(cpu_model)
    dummy_ids  = torch.zeros(1, BERT_MAX_LEN, dtype=torch.long)
    dummy_mask = torch.ones(1, BERT_MAX_LEN,  dtype=torch.long)
    path = os.path.join(SAVE_DIR, f'{name}.onnx')
    torch.onnx.export(
        wrapper,
        (dummy_ids, dummy_mask),
        path,
        input_names=['input_ids', 'attention_mask'],
        output_names=['binary_logits', 'ml_logits'],
        dynamic_axes={
            'input_ids':      {0: 'batch_size'},
            'attention_mask': {0: 'batch_size'},
            'binary_logits':  {0: 'batch_size'},
            'ml_logits':      {0: 'batch_size'},
        },
        opset_version=14,
    )
    model.to(DEVICE)
    print(f'  ONNX saved → {path}')

print('\nExporting to ONNX…')
export_onnx(mtl_model, 'multitask_model')
print('Done.')


Weights saved to models\trained/
Thresholds saved: [0.4645569620253165, 0.1, 0.29240506329113924, 0.5860759493670886]

Exporting to ONNX…


W0328 12:27:54.733000 23080 site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0328 12:27:55.300000 23080 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0328 12:27:55.302000 23080 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_sc

[torch.onnx] Obtain model graph for `_OnnxWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `_OnnxWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\Wassim VQ\AppData\Local\Programs\Python\Python311\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Wassim VQ\AppData\Local\Programs\Python\Python311\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "c:\Users\Wassim VQ\AppData\Local\Programs\Python\Python311\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Wassim VQ\AppData\Local\Programs\Python\Python311\Lib\site-packages\onnx\version_converter.py", line 37, in convert_version


Applied 56 of general pattern rewrite rules.
  ONNX saved → models\trained\multitask_model.onnx
Done.



## 6. Evaluation on RAG-Generated Memes


In [11]:

# ══════════════════════════════════════════════════════════════════════════════
# 6. EVALUATION ON RAG-GENERATED MEMES
# ══════════════════════════════════════════════════════════════════════════════

GEN_CSV = os.path.join('evaluation', 'results', 'generated_memes_rag.csv')
gen_df  = pd.read_csv(GEN_CSV, sep='\t', header=0)

for col in fixed_cols:
    assert col in gen_df.columns, f'Generated CSV missing expected column: {col}'

gen_text_cols = [c for c in gen_df.columns if c not in fixed_cols]
gen_df['Text'] = gen_df[gen_text_cols].astype(str).agg(' '.join, axis=1)

gen_texts      = gen_df['Text'].tolist()
gen_binary_lbl = gen_df['misogynous'].tolist()
gen_ml_labels  = gen_df[label_cols].values.tolist()

# ── Cascaded evaluation on generated memes ───────────────────────────────────
gen_bin_preds, gen_ml_preds = cascaded_predict(
    mtl_model, gen_texts, ml_thresholds
)

gen_bin_f1 = f1_score(gen_binary_lbl, gen_bin_preds, average='macro')
gen_ml_f1  = f1_score(gen_ml_labels, gen_ml_preds, average='macro')

print('── Generated Memes — Cascaded Pipeline Evaluation ──────────────────────')
print(f'  Binary F1-Macro      : {gen_bin_f1:.4f}  ({len(gen_texts)} samples)')
print(f'  E2E ML F1-Macro      : {gen_ml_f1:.4f}')
print()
print('Stage 1 — Binary:')
print(classification_report(
    gen_binary_lbl, gen_bin_preds,
    target_names=['non-misogynous', 'misogynous']
))
print('End-to-End — Multi-label (cascaded):')
print(classification_report(
    gen_ml_labels, gen_ml_preds,
    target_names=label_cols
))


── Generated Memes — Cascaded Pipeline Evaluation ──────────────────────
  Binary F1-Macro      : 0.8824  (50 samples)
  E2E ML F1-Macro      : 0.4291

Stage 1 — Binary:
                precision    recall  f1-score   support

non-misogynous       0.67      1.00      0.80         6
    misogynous       1.00      0.93      0.96        44

      accuracy                           0.94        50
     macro avg       0.83      0.97      0.88        50
  weighted avg       0.96      0.94      0.94        50

End-to-End — Multi-label (cascaded):
                 precision    recall  f1-score   support

        shaming       0.33      0.09      0.14        11
     stereotype       0.24      0.91      0.38        11
objectification       0.40      0.55      0.46        11
       violence       0.73      0.73      0.73        11

      micro avg       0.36      0.57      0.44        44
      macro avg       0.43      0.57      0.43        44
   weighted avg       0.43      0.57      0.43       


## 7. Results Comparison & Error Analysis


In [12]:

# ══════════════════════════════════════════════════════════════════════════════
# 7. RESULTS COMPARISON & ERROR ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

# ── Failure type heuristics (for error-analysis section) ──────────────────────
_NEG_RE  = re.compile(r"\b(not|never|no|n't|without|barely|hardly|neither|nor)\b", re.I)
_IRON_RE = re.compile(r"\b(obviously|clearly|apparently|sure|just|totally|definitely|righ+t)\b", re.I)
_HEDG_RE = re.compile(r"\b(maybe|perhaps|kind of|sort of|almost|quite|rather|could be)\b", re.I)

def _failure_tags(text):
    tags = []
    if len(text.split()) < 8:
        tags.append('SHORT_TEXT')
    if _NEG_RE.search(text):
        tags.append('NEGATION')
    if _IRON_RE.search(text):
        tags.append('IRONY/SARCASM')
    if _HEDG_RE.search(text):
        tags.append('AMBIGUOUS_HEDGE')
    return tags or ['OTHER']

# ── Baseline vs. Multi-Task — Comparison Table ───────────────────────────────
model_label = 'MTL+FGM'
col_w = max(len(model_label), 12)
print(f'{"─"*(30 + col_w)}')
print(f'{"Task":<26} {"TF-IDF+LR":>10} {model_label:>{col_w}} {"Gain":>8}')
print(f'{"─"*(30 + col_w)}')
bin_gain = cascaded_bin_f1 - baseline_bin_f1
ml_gain  = cascaded_ml_f1  - baseline_ml_f1
print(f'{"Binary":<26} {baseline_bin_f1:>10.4f} {cascaded_bin_f1:>{col_w}.4f} {bin_gain:>+8.4f}')
print(f'{"Multi-label (E2E casc.)":<26} {baseline_ml_f1:>10.4f} {cascaded_ml_f1:>{col_w}.4f} {ml_gain:>+8.4f}')
print(f'{"─"*(30 + col_w)}')

# ── Top 20 Worst Failures — Binary ───────────────────────────────────────────
lbl_map = {0: 'non-misogynous', 1: 'misogynous'}

failures_bin = [
    {
        'text': X_test_list[i],
        'true': lbl_map[t],
        'pred': lbl_map[p],
        'tags': _failure_tags(X_test_list[i]),
    }
    for i, (t, p) in enumerate(zip(y_bin_test_list, cascade_bin_preds)) if t != p
]

bin_tag_counts = Counter(tag for f in failures_bin for tag in f['tags'])
print(f'\n── Top 20 Binary Failures  (total wrong: {len(failures_bin)} / {len(y_bin_test_list)}) ──')
print(f'  Failure types: {dict(bin_tag_counts)}')
for j, f in enumerate(failures_bin[:20], 1):
    snippet = f['text'][:100].replace('\n', ' ')
    print(f'{j:2d}. TRUE={f["true"]:>15s}  PRED={f["pred"]:>15s}  TAGS={f["tags"]}')
    print(f'    "{snippet}…"')

# ── Top 20 Worst Failures — Multi-label (cascaded) ───────────────────────────
def _n_errors(t, p): return sum(ti != pi for ti, pi in zip(t, p))

failures_ml = sorted([
    {
        'text':  X_test_list[i],
        'true':  [label_cols[k] for k, v in enumerate(y_ml_test_list[i]) if v],
        'pred':  [label_cols[k] for k, v in enumerate(cascade_ml_preds[i]) if v],
        'n_err': _n_errors(y_ml_test_list[i], cascade_ml_preds[i]),
        'tags':  _failure_tags(X_test_list[i]),
    }
    for i in range(len(y_ml_test_list))
    if _n_errors(y_ml_test_list[i], cascade_ml_preds[i]) > 0
], key=lambda x: -x['n_err'])

ml_tag_counts = Counter(tag for f in failures_ml for tag in f['tags'])
print(f'\n── Top 20 Multi-label Failures  (total wrong: {len(failures_ml)} / {len(y_ml_test_list)}) ──')
print(f'  Failure types: {dict(ml_tag_counts)}')
for j, f in enumerate(failures_ml[:20], 1):
    snippet = f['text'][:100].replace('\n', ' ')
    print(f'{j:2d}. [{f["n_err"]} err]  TRUE={f["true"]}  PRED={f["pred"]}  TAGS={f["tags"]}')
    print(f'    "{snippet}…"')


──────────────────────────────────────────
Task                        TF-IDF+LR      MTL+FGM     Gain
──────────────────────────────────────────
Binary                         0.7879       0.8253  +0.0374
Multi-label (E2E casc.)        0.4829       0.5199  +0.0370
──────────────────────────────────────────

── Top 20 Binary Failures  (total wrong: 262 / 1500) ──
  Failure types: {'OTHER': 180, 'NEGATION': 28, 'IRONY/SARCASM': 16, 'AMBIGUOUS_HEDGE': 3, 'SHORT_TEXT': 38}
 1. TRUE= non-misogynous  PRED=     misogynous  TAGS=['OTHER']
    "SOMETIMES I WISH I COULD AFFORD A HOOKER FOR AN ENTIRE NIGHT ONLY GIVE HER DECENT CLOTHING, TAKE HER…"
 2. TRUE= non-misogynous  PRED=     misogynous  TAGS=['OTHER']
    "PICTOPHLE APP *Straight white malle starts talking *Puts headphones in â˜† ears Feminism â™¡1 Reply …"
 3. TRUE= non-misogynous  PRED=     misogynous  TAGS=['OTHER']
    "How is it, that you always have enough money? Im already broke again! *le derp That sounds horrible!…"
 4. TRUE=   